# OpenLux Hub — 01 Quickstart : Accès aux Données & API

Ce notebook présente les différentes manières de consommer les données ouvertes consolidées d'OpenLux :
1. **Requêtes SQL DuckDB directes** sur les fichiers Parquet Gold (ZSTD) en local sans serveur.
2. **API REST FastAPI** (`/v1/datasets`, `/v1/data/{dataset}`, streaming CSV & GeoJSON).
3. **Métadonnées MLOps & Gouvernance** (`_meta`, `trust_score`, traçabilité SHA-256).
4. **Standards Open Data** (Frictionless `datapackage.json` et profils européens NeTEx / SIRI / DATEX II).

## 1. Requêtage Direct DuckDB sur les Parquets Gold

DuckDB permet d'interroger directement les fichiers Parquet compressés en ZSTD avec des performances maximales et sans démarrer de base de données.

In [ ]:
import duckdb

# Connexion in-memory DuckDB
con = duckdb.connect()

# Lecture directe du Data Mart Météo
df_weather = con.execute("""
    SELECT station_id, station_name, temperature_c, humidity_percent, observed_at_utc
    FROM 'data/gold/gold_weather_observations.parquet'
    ORDER BY observed_at_utc DESC
    LIMIT 5
""").df()

df_weather

## 2. Exploration des Tables de Mobilité & Transports

OpenLux applique les standards sectoriels européens : **DATEX II** pour les événements autoroutiers, **NeTEx / SIRI** pour le transport en commun, et horaires d'ouverture standardisés (`24/7`).

In [ ]:
# Aperçu des parkings et stations de recharge au Luxembourg
df_mobility = con.execute("""
    SELECT facility_name, facility_type, total_capacity, available_spaces, opening_hours, commune
    FROM 'data/gold/gold_mobility_parkings.parquet'
    LIMIT 5
""").df()

df_mobility

## 3. Consommation via l'API REST FastAPI

L'API OpenLux permet d'extraire les données en JSON, CSV streamé ou GeoJSON pour les outils SIG.

In [ ]:
# Exemple d'appel API REST :
# curl http://127.0.0.1:8000/v1/data/gold_fuel_prices?limit=3
# Ou via httpx en Python :
# import httpx
# res = httpx.get('http://127.0.0.1:8000/v1/data/gold_fuel_prices?limit=3')
# print(res.json()['_meta']['trust_score'])

## 4. Consultation des Métadonnées Open Data (`datapackage.json`)

OpenLux génère automatiquement un manifeste Frictionless Data Package conforme aux standards de `data.public.lu`.

In [ ]:
import json
from pathlib import Path

dp_path = Path("data/gold/datapackage.json")
if dp_path.exists():
    dp = json.loads(dp_path.read_text(encoding="utf-8"))
    print(f"Titre : {dp['title']}")
    print(f"Licence : {dp['license']}")
    print(f"Nombre de Data Marts : {len(dp['resources'])}")
    for r in dp["resources"][:3]:
        print(f" - {r['name']} ({r['row_count']} lignes, standards : {r['standards_applied']})")